# Tutorial 3: Function Secret Sharing
在我们的库中，我们还支持函数秘密分享，这部分内容参考了Boyle等人的论文：Function Secret Sharing: Improvements and Extensions.2016；Function Secret Sharing for Mixed-Mode and Fixed-Point Secure Computation.2021
函数秘密分享主要是三个函数的秘密分享，分布式点函数(DPF)，分布式比较函数(DCF)，分布式区间函数(DICF)。

## 分布式点函数
定义如下：
$$
f_{\alpha,\beta}(x)=\begin{cases}
\beta, & \text {if x=$\alpha$} \\
0, & \text {else}
\end{cases}
$$
我们需要求函数在x位置的值，同时需要隐藏$\alpha$，$\beta$，我们采取的方案是通过可信第三方生成关于$\alpha$，$\beta$的DPF密钥，再将隐藏$\alpha$，$\beta$信息的密钥分发给两个参与方，两个参与方各自利用密钥求当自变量为x时函数的值。

In [1]:
# import the libraries
import torch
from crypto.primitives.function_secret_sharing.dpf import DPF
from crypto.tensor.RingTensor import RingTensor

num_of_keys = 10  # We need a few keys for a few function values, but of course we can generate many keys in advance.

# generate keys in offline phase
# set alpha and beta
alpha = RingTensor.convert_to_ring(torch.tensor(5))
beta = RingTensor.convert_to_ring(torch.tensor(1))

Key0, Key1 = DPF.gen(num_of_keys=num_of_keys, alpha=alpha, beta=beta)

# online phase
# generate some values what we need to evaluate
x = RingTensor.convert_to_ring(torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))

# Party 0:
res_0 = DPF.eval(x=x, keys=Key0, party_id=0)

# Party 1:
res_1 = DPF.eval(x=x, keys=Key1, party_id=1)

# restore result
res = res_0 + res_1
print(res)

tensor([0, 0, 0, 0, 1, 0, 0, 0, 0, 0], device='cuda:0')


## 分布式比较函数
定义如下：
$$
f_{\alpha,\beta}(x)=\begin{cases}
\beta, & \text {if x < $\alpha$} \\
0, & \text {else}
\end{cases}
$$
求DCF值的方法和DPF类似

In [2]:
# import the libraries
import torch
from crypto.primitives.function_secret_sharing.dcf import DCF
from crypto.tensor.RingTensor import RingTensor

num_of_keys = 10  # We need a few keys for a few function values, but of course we can generate many keys in advance.

# generate keys in offline phase
# set alpha and beta
alpha = RingTensor.convert_to_ring(torch.tensor(5))
beta = RingTensor.convert_to_ring(torch.tensor(1))

Key0, Key1 = DCF.gen(num_of_keys=num_of_keys, alpha=alpha, beta=beta)

# online phase
# generate some values what we need to evaluate
x = RingTensor.convert_to_ring(torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))

# Party 0:
res_0 = DCF.eval(x=x, keys=Key0, party_id=0)

# Party 1:
res_1 = DCF.eval(x=x, keys=Key1, party_id=1)

# restore result
res = res_0 + res_1
print(res)

tensor([1, 1, 1, 1, 0, 0, 0, 0, 0, 0], device='cuda:0')


## 分布式区间函数
定义如下：
$$
f_{p,q}(x)=\begin{cases}
1, & \text {if p$\leq$x $\leq$ q} \\
0, & \text {else}
\end{cases}
$$
我们设定当x出现在p, q区间里，那么函数值就为1，通过DICF可以实现两个数之间的大小比较，对于两个数x, y，对于其差值，如果在环可表示正数区间内，就说明x大于y。
现在我们将演示对于给定输入x，如何求分布式区间函数的值，不过和DPF, DCF有所不同的是，DICF隐藏的输入x的信息，而非上下界的信息，这也是为了大小比较服务。

In [3]:
# import the libraries
# import the libraries
import torch
from crypto.primitives.function_secret_sharing.dicf import DICF
from crypto.tensor.RingTensor import RingTensor
from config.base_configs import DEVICE

# generate key in offline phase
num_of_keys = 10
down_bound = torch.tensor([3]).to(DEVICE)
upper_bound = torch.tensor([7]).to(DEVICE)

Key0, Key1 = DICF.gen(num_of_keys=num_of_keys, down_bound=down_bound, upper_bound=upper_bound)

# evaluate x in online phase
# generate some values what we need to evaluate
x = RingTensor.convert_to_ring(torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))
x_shift = x + Key0.r.reshape(x.shape) + Key1.r.reshape(x.shape)

# online phase
# Party 0:
res_0 = DICF.eval(x_shift=x_shift, keys=Key0, party_id=0, down_bound=down_bound, upper_bound=upper_bound)

# Party 1:
res_1 = DICF.eval(x_shift=x_shift, keys=Key1, party_id=1, down_bound=down_bound, upper_bound=upper_bound)

# restore result
res = res_0 + res_1
print(res)

tensor([0, 0, 1, 1, 1, 1, 1, 0, 0, 0], device='cuda:0')


## 函数秘密分享的应用
之前也提到，DICF可以用于大小比较，如果想要利用DICF进行大小比较，请修改./config/base_configs.py中的GE_TYPE，将其修改为FSS，然后参考Tutorial_2中的方法即可实现。我们在Tutorial_0中也讲过有一种改进的FSS方法，如果想要用这种方法大小比较的话，将GE_TYPE修改为GROTTO然后参考Tutorial_2。